In [1]:
!pip install networkx torch torch-geometric matplotlib seaborn scipy plotly watchdog ipywidgets
!pip install nvdlib requests python-dateutil scikit-learn pandas numpy
!pip install dash dash-bootstrap-components dash-cytoscape

# Enable Jupyter widgets (for GUI)
!jupyter nbextension enable --py widgetsnbextension

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 57.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 44.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 110.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 14.8 MB/s eta 0:00:00
  Created wheel for dash-cytoscape: filename=dash_cytoscape-1.0.2-py3-none-any.whl size=4010717 sha256=cb25b56d88b2d589315f9045ccfddd2e28f8e238c239dc4fb9735d8b80bb5c67
  Stored in directory: /root/.cache/pip/wheels/0c/db/f6/9dcb225e9adf45dfef713542769556b1f508170a0759053892
Successfully built dash-cytoscape
Enabling notebook extension jupyter-js-widgets/extension...
      - Validating: OK


In [2]:
# ===========================
# Universal Log Anomaly Detection - All-in-one
# Paste this entire block into a Jupyter / Colab cell and run.
#
# NOTE for Colab: run the "INSTALL DEPENDENCIES" block below in a separate cell first
# if you don't already have the packages installed.
# ===========================

# ---------------------------
# INSTALL DEPENDENCIES (Colab/Jupyter)
# ---------------------------
# If you're in Colab or a clean environment, run only this small block first:
#
# !pip install networkx torch torch-geometric matplotlib seaborn scipy plotly watchdog ipywidgets
# !pip install nvdlib requests python-dateutil scikit-learn pandas numpy
# !pip install dash dash-bootstrap-components dash-cytoscape
# !jupyter nbextension enable --py widgetsnbextension
#
# Then run the rest of this big code cell.

# ---------------------------
# BEGIN FULL SYSTEM CODE
# ---------------------------

import re
import json
import os
import time
import hashlib
from datetime import datetime, timedelta
from collections import defaultdict, Counter
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Optional imports that may be heavy; guard them where used
try:
    import torch
    import torch.nn.functional as F
    from torch_geometric.data import Data
    from torch_geometric.nn import GCNConv, global_mean_pool
except Exception:
    torch = None

try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    import plotly.express as px
except Exception:
    go = None
    make_subplots = None
    px = None

try:
    from scipy import stats
    from sklearn.cluster import DBSCAN
    from sklearn.preprocessing import StandardScaler
    from sklearn.ensemble import IsolationForest
except Exception:
    stats = None
    DBSCAN = None
    StandardScaler = None
    IsolationForest = None

# NVD library and requests
try:
    import requests
    import nvdlib
    from dateutil.parser import parse as parse_date
except Exception:
    nvdlib = None
    requests = None
    parse_date = None

# ---------------------------
# NVDScorer
# ---------------------------
class NVDScorer:
    """NVD-based vulnerability scoring system"""

    def __init__(self, cache_file="nvd_cache.json"):
        self.cache_file = cache_file
        self.vulnerability_cache = self.load_cache()
        self.attack_patterns = self.initialize_attack_patterns()
        self.scoring_weights = {
            'critical': 10.0,
            'high': 7.5,
            'medium': 5.0,
            'low': 2.5,
            'none': 0.0
        }

    def load_cache(self):
        """Load cached vulnerability data"""
        try:
            with open(self.cache_file, 'r') as f:
                return json.load(f)
        except Exception:
            return {}

    def save_cache(self):
        """Save vulnerability cache"""
        try:
            with open(self.cache_file, 'w') as f:
                json.dump(self.vulnerability_cache, f, indent=2)
        except Exception as e:
            print(f"Failed to save NVD cache: {e}")

    def initialize_attack_patterns(self):
        """Initialize attack pattern detection"""
        return {
            'brute_force': re.compile(r'(failed|authentication|login|ssh|password)', re.IGNORECASE),
            'privilege_escalation': re.compile(r'(sudo|root|privilege|escalation|setuid)', re.IGNORECASE),
            'network_intrusion': re.compile(r'(connection|refused|timeout|firewall|iptables)', re.IGNORECASE),
            'malware': re.compile(r'(virus|malware|trojan|suspicious|quarantine)', re.IGNORECASE),
            'dos_attack': re.compile(r'(denial|service|flood|overwhelmed|capacity)', re.IGNORECASE),
            'data_exfiltration': re.compile(r'(transfer|download|export|copy|unauthorized)', re.IGNORECASE),
            'system_compromise': re.compile(r'(compromised|breach|unauthorized|access|intrusion)', re.IGNORECASE),
            'configuration_error': re.compile(r'(misconfigured|configuration|settings|invalid)', re.IGNORECASE)
        }

    def detect_attack_type(self, log_message):
        """Detect potential attack type from log message"""
        detected_attacks = []
        for attack_type, pattern in self.attack_patterns.items():
            if pattern.search(log_message):
                detected_attacks.append(attack_type)
        return detected_attacks or ['unknown']

    def get_nvd_score(self, attack_types, context_info=None):
        """Get NVD-based scoring for detected attack types"""
        max_score = 0.0
        severity = 'none'

        for attack_type in attack_types:
            cache_key = f"{attack_type}_{hashlib.md5(str(context_info).encode()).hexdigest()[:8]}"

            if cache_key in self.vulnerability_cache:
                cached_result = self.vulnerability_cache[cache_key]
                score = cached_result.get('score', 0.0)
                sev = cached_result.get('severity', 'none')
            else:
                score, sev = self.query_nvd_for_attack(attack_type, context_info)
                self.vulnerability_cache[cache_key] = {
                    'score': score,
                    'severity': sev,
                    'timestamp': datetime.now().isoformat()
                }

            if score > max_score:
                max_score = score
                severity = sev

        return max_score, severity

    def query_nvd_for_attack(self, attack_type, context_info=None):
        """Query NVD database for attack-related CVEs"""
        try:
            if nvdlib is None:
                # No NVD library available; return fallback
                return self._fallback_for(attack_type)

            keyword_mapping = {
                'brute_force': 'authentication bypass',
                'privilege_escalation': 'privilege escalation',
                'network_intrusion': 'remote code execution',
                'malware': 'malicious code',
                'dos_attack': 'denial of service',
                'data_exfiltration': 'information disclosure',
                'system_compromise': 'unauthorized access',
                'configuration_error': 'configuration'
            }

            keyword = keyword_mapping.get(attack_type, attack_type)
            end_date = datetime.utcnow()
            start_date = end_date - timedelta(days=365)
            # Use nvdlib search; if it errors fallback
            cves = []
            try:
                cves = nvdlib.searchCVE(
                    keywordSearch=keyword,
                    pubStartDate=start_date,
                    pubEndDate=end_date,
                    limit=10
                )
            except Exception as e:
                # nvdlib may fail in offline environments
                # print(f"NVD search failed: {e}")
                return self._fallback_for(attack_type)

            if cves:
                scores = []
                severities = []
                for cve in cves:
                    if hasattr(cve, 'metrics') and cve.metrics:
                        for metric in cve.metrics:
                            if hasattr(metric, 'cvssData'):
                                try:
                                    score = metric.cvssData.baseScore
                                    scores.append(score)
                                    if score >= 9.0:
                                        severities.append('critical')
                                    elif score >= 7.0:
                                        severities.append('high')
                                    elif score >= 4.0:
                                        severities.append('medium')
                                    else:
                                        severities.append('low')
                                except Exception:
                                    continue
                if scores:
                    avg_score = float(np.mean(scores))
                    most_common_severity = Counter(severities).most_common(1)[0][0] if severities else 'none'
                    return avg_score, most_common_severity

            return self._fallback_for(attack_type)

        except Exception as e:
            # print(f"NVD query error for {attack_type}: {e}")
            return self._fallback_for(attack_type)

    def _fallback_for(self, attack_type):
        fallback_scores = {
            'brute_force': (6.5, 'medium'),
            'privilege_escalation': (8.5, 'high'),
            'network_intrusion': (9.0, 'critical'),
            'malware': (8.0, 'high'),
            'dos_attack': (7.0, 'high'),
            'data_exfiltration': (7.5, 'high'),
            'system_compromise': (9.5, 'critical'),
            'configuration_error': (4.0, 'medium'),
            'unknown': (2.0, 'low')
        }
        return fallback_scores.get(attack_type, (2.0, 'low'))


# ---------------------------
# AdaptiveThresholdDetector
# ---------------------------
class AdaptiveThresholdDetector:
    """Adaptive threshold detection using multiple statistical methods"""

    def __init__(self, adaptation_window=1000, min_samples=50):
        self.adaptation_window = adaptation_window
        self.min_samples = min_samples
        self.historical_data = []
        self.current_threshold = None
        self.threshold_history = []
        self.outlier_detectors = {}
        try:
            self.outlier_detectors['isolation_forest'] = IsolationForest(contamination=0.1, random_state=42)
            self.outlier_detectors['dbscan'] = DBSCAN(eps=0.5, min_samples=5)
        except Exception:
            self.outlier_detectors['isolation_forest'] = None
            self.outlier_detectors['dbscan'] = None
        try:
            self.scaler = StandardScaler()
        except Exception:
            self.scaler = None

    def update_data(self, new_data):
        """Update historical data with new observations"""
        if isinstance(new_data, (list, np.ndarray)):
            self.historical_data.extend(list(new_data))
        else:
            self.historical_data.append(new_data)

        # Maintain sliding window
        if len(self.historical_data) > self.adaptation_window:
            self.historical_data = self.historical_data[-self.adaptation_window:]

    def compute_adaptive_threshold(self, method='ensemble'):
        """Compute adaptive threshold using multiple methods"""
        if len(self.historical_data) < self.min_samples:
            # Use conservative default until we have enough data
            self.current_threshold = 2.0
            return self.current_threshold

        data = np.array(self.historical_data).reshape(-1, 1)

        if method == 'ensemble':
            thresholds = []

            # Method 1: Statistical (Modified Z-score)
            try:
                median = np.median(self.historical_data)
                mad = np.median(np.abs(np.array(self.historical_data) - median))
                if mad == 0:
                    stat_threshold = 2.0
                else:
                    modified_z_scores = 0.6745 * (np.array(self.historical_data) - median) / mad
                    stat_threshold = np.percentile(np.abs(modified_z_scores), 95)
                thresholds.append(stat_threshold)
            except Exception:
                thresholds.append(2.0)

            # Method 2: Isolation Forest
            try:
                iso = self.outlier_detectors.get('isolation_forest')
                if iso is not None:
                    iso.fit(data)
                    scores = iso.decision_function(data)
                    iso_threshold = np.percentile(scores, 5)
                    thresholds.append(abs(iso_threshold) * 3)
                else:
                    thresholds.append(2.0)
            except Exception:
                thresholds.append(2.0)

            # Method 3: DBSCAN-based
            try:
                db = self.outlier_detectors.get('dbscan')
                if db is not None and self.scaler is not None:
                    scaled_data = self.scaler.fit_transform(data)
                    clustering = db.fit(scaled_data)
                    noise_points = np.sum(clustering.labels_ == -1)
                    noise_ratio = noise_points / len(data)
                    dbscan_threshold = 2.0 + (noise_ratio * 3)
                    thresholds.append(dbscan_threshold)
                else:
                    thresholds.append(2.0)
            except Exception:
                thresholds.append(2.0)

            # Method 4: Percentile-based
            try:
                percentile_threshold = (np.percentile(self.historical_data, 95) -
                                      np.percentile(self.historical_data, 50)) / np.std(self.historical_data)
                thresholds.append(max(1.5, percentile_threshold))
            except Exception:
                thresholds.append(2.0)

            weights = [0.3, 0.25, 0.25, 0.2]
            try:
                self.current_threshold = float(np.average(thresholds, weights=weights))
            except Exception:
                self.current_threshold = 2.0

        elif method == 'statistical':
            try:
                data_array = np.array(self.historical_data)
                q1, q3 = np.percentile(data_array, [25, 75])
                iqr = q3 - q1
                skewness = stats.skew(data_array) if stats is not None else 0
                kurtosis = stats.kurtosis(data_array) if stats is not None else 0
                if abs(skewness) > 1 or kurtosis > 3:
                    multiplier = 2.0
                else:
                    multiplier = 1.5
                lower_bound = q1 - multiplier * iqr
                upper_bound = q3 + multiplier * iqr
                mean_val = np.mean(data_array)
                std_val = np.std(data_array)
                if std_val > 0:
                    self.current_threshold = abs(upper_bound - mean_val) / std_val
                else:
                    self.current_threshold = 2.0
            except Exception:
                self.current_threshold = 2.0

        self.current_threshold = max(1.2, min(5.0, float(self.current_threshold)))
        self.threshold_history.append({
            'threshold': self.current_threshold,
            'timestamp': datetime.now().isoformat(),
            'sample_size': len(self.historical_data)
        })
        return self.current_threshold

    def detect_anomalies(self, new_data):
        """Detect anomalies in new data using adaptive threshold"""
        if self.current_threshold is None:
            self.compute_adaptive_threshold()

        if len(self.historical_data) < 2:
            return [], []

        mean_val = np.mean(self.historical_data)
        std_val = np.std(self.historical_data)
        if std_val == 0:
            return [], []

        z_scores = [(x - mean_val) / std_val for x in new_data]
        anomalies = [i for i, z in enumerate(z_scores) if abs(z) > self.current_threshold]
        return anomalies, z_scores


# ---------------------------
# UniversalLogAnalyzer
# ---------------------------
class UniversalLogAnalyzer:
    """Universal log analyzer with NVD integration and adaptive thresholding"""

    def __init__(self, enable_realtime=False):
        self.enable_realtime = enable_realtime
        self.G = nx.DiGraph()
        self.nodes_set = set()
        self.node_types = {}
        self.edge_counts = Counter()
        self.edge_weights = defaultdict(float)
        self.nvd_scorer = NVDScorer()
        self.adaptive_detector = AdaptiveThresholdDetector()
        self.anomalies = []
        self.processed_logs = []
        self.attack_intelligence = defaultdict(list)
        self.pattern_registry = {}
        self.auto_patterns = set()
        self.metadata = {
            'processing_time': None,
            'total_nodes': 0,
            'total_edges': 0,
            'anomalies_detected': 0,
            'nvd_queries': 0,
            'adaptive_threshold': None,
            'last_updated': None
        }
        self._initialize_universal_patterns()

    def _initialize_universal_patterns(self):
        """Initialize universal log patterns that adapt to any log format"""
        self.pattern_registry = {
            'timestamp': [
                re.compile(r'\[([\d.]+)\]'),
                re.compile(r'(\d{4}-\d{2}-\d{2}[T ]\d{2}:\d{2}:\d{2})'),
                re.compile(r'(\w{3}\s+\d{1,2}\s+\d{2}:\d{2}:\d{2})'),
            ],
            'process': [
                re.compile(r'([a-zA-Z0-9_\-]+)\[\d+\]'),
                re.compile(r'([a-zA-Z0-9_\-]+):\s'),
                re.compile(r'<\d+>.*?\s+([a-zA-Z0-9_\-]+)'),
            ],
            'severity': [
                re.compile(r'\b(EMERG|ALERT|CRIT|ERR|WARNING|NOTICE|INFO|DEBUG)\b', re.IGNORECASE),
                re.compile(r'\b(FATAL|ERROR|WARN|INFO|DEBUG|TRACE)\b', re.IGNORECASE),
                re.compile(r'\b(CRITICAL|HIGH|MEDIUM|LOW)\b', re.IGNORECASE),
            ],
            'network': [
                re.compile(r'\b(\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3})\b'),
                re.compile(r'\b([a-zA-Z0-9_\-]+\.[a-zA-Z0-9_\-]+\.[a-zA-Z]{2,})\b'),
                re.compile(r'\b(port|PORT)\s+(\d+)\b'),
            ],
            'user': [
                re.compile(r'\buser[:\s]+([a-zA-Z0-9_\-]+)\b', re.IGNORECASE),
                re.compile(r'\b(root|admin|administrator)\b', re.IGNORECASE),
                re.compile(r'/home/([^/\s]+)'),
            ],
            'service': [
                re.compile(r'\b(started|stopped|failed|restarted)\s+([a-zA-Z0-9_\-\.]+)', re.IGNORECASE),
                re.compile(r'systemd.*?([a-zA-Z0-9_\-\.]+)\.service', re.IGNORECASE),
            ]
        }

    def auto_discover_patterns(self, log_lines):
        """Automatically discover new patterns in log data"""
        common_tokens = Counter()
        for line in log_lines[:100]:
            tokens = re.findall(r'[a-zA-Z_][a-zA-Z0-9_]*', line)
            common_tokens.update(tokens)
        for token, count in common_tokens.most_common(20):
            if count > 5 and len(token) > 2:
                pattern = re.compile(rf'\b{re.escape(token)}\b', re.IGNORECASE)
                if 'auto_discovered' not in self.pattern_registry:
                    self.pattern_registry['auto_discovered'] = []
                self.pattern_registry['auto_discovered'].append(pattern)
                self.auto_patterns.add(token)

    def extract_entities(self, log_line):
        """Extract all entities from a log line using universal patterns"""
        entities = defaultdict(list)
        for category, patterns in self.pattern_registry.items():
            for pattern in patterns:
                matches = pattern.findall(log_line)
                if matches:
                    entities[category].extend(matches if isinstance(matches, list) else [matches])
        return entities

    def calculate_nvd_weight(self, source, target, log_line, entities):
        """Calculate edge weight using NVD scoring"""
        attack_types = self.nvd_scorer.detect_attack_type(log_line) if self.nvd_scorer else ['unknown']
        context_info = {
            'source': source,
            'target': target,
            'entities': dict(entities),
            'log_snippet': log_line[:100]
        }
        try:
            nvd_score, severity = self.nvd_scorer.get_nvd_score(attack_types, context_info) if self.nvd_scorer else (0.0, 'none')
        except Exception:
            nvd_score, severity = 0.0, 'none'
        if attack_types != ['unknown']:
            self.attack_intelligence[source].append({
                'attack_types': attack_types,
                'severity': severity,
                'score': nvd_score,
                'timestamp': datetime.now().isoformat(),
                'target': target,
                'log_line': log_line
            })
        base_weight = 1.0
        severity_multipliers = {
            'critical': 10.0,
            'high': 5.0,
            'medium': 2.0,
            'low': 1.0,
            'none': 0.5
        }
        severity_weight = severity_multipliers.get(severity, 1.0)
        entity_weight = 1.0
        if entities.get('severity'):
            entity_weight *= 1.5
        if entities.get('network'):
            entity_weight *= 1.2
        if entities.get('user') and any('root' in str(u).lower() for u in entities.get('user', [])):
            entity_weight *= 2.0
        final_weight = base_weight * severity_weight * entity_weight * (1 + nvd_score / 10.0)
        return final_weight, nvd_score, severity, attack_types

    def process_log_line(self, log_line):
        """Process a single log line universally"""
        entities = self.extract_entities(log_line)
        source = None
        targets = []
        if entities.get('process'):
            # pattern.findall may return tuples; normalize
            src = entities['process'][0]
            if isinstance(src, tuple):
                src = ''.join([s for s in src if s])
            source = src
        elif entities.get('service'):
            src = entities['service'][0]
            if isinstance(src, tuple):
                src = ''.join([s for s in src if s])
            source = src
        elif 'kernel' in log_line.lower():
            source = 'kernel'
        elif 'system' in log_line.lower():
            source = 'system'
        else:
            source = 'unknown'
        if entities.get('user'):
            targets.extend(entities['user'])
        if entities.get('network'):
            # network matches might be tuples; flatten
            for net in entities['network']:
                if isinstance(net, tuple):
                    # try to pick an IP-like item
                    for item in net:
                        if re.match(r'\d+\.\d+\.\d+\.\d+', str(item)):
                            targets.append(f"network_{item}")
                            break
                else:
                    targets.append(f"network_{net}")
        if entities.get('service'):
            for s in entities['service']:
                if isinstance(s, tuple):
                    # choose last element (service name)
                    s2 = next((x for x in s if x and not x.isdigit()), ''.join(s))
                    if s2 != source:
                        targets.append(s2)
                else:
                    if s != source:
                        targets.append(s)
        self.add_node(source, 'process' if entities.get('process') else 'service')
        for target in targets:
            if not target or target == source:
                continue
            target_type = 'user' if any(str(target).lower().startswith('user') or str(target).lower().startswith('root') for target in [target]) else 'service'
            self.add_node(target, target_type)
            weight, nvd_score, severity, attack_types = self.calculate_nvd_weight(source, target, log_line, entities)
            self.add_edge(source, target, weight, nvd_score, severity, attack_types)
        self.processed_logs.append({
            'source': source,
            'targets': targets,
            'entities': dict(entities),
            'timestamp': datetime.now().isoformat(),
            'raw_log': log_line
        })

    def add_node(self, node, node_type="other"):
        """Add node to graph with type"""
        if node and node not in self.node_types:
            self.node_types[node] = node_type
            self.nodes_set.add(node)

    def add_edge(self, src, tgt, weight=1.0, nvd_score=0.0, severity='none', attack_types=None):
        """Add edge with comprehensive weight information"""
        if src and tgt and src != tgt:
            edge_key = (src, tgt)
            self.edge_counts[edge_key] += 1
            self.edge_weights[edge_key] += weight
            if not hasattr(self, 'edge_metadata'):
                self.edge_metadata = defaultdict(list)
            self.edge_metadata[edge_key].append({
                'nvd_score': nvd_score,
                'severity': severity,
                'attack_types': attack_types or [],
                'timestamp': datetime.now().isoformat()
            })

    def build_graph(self):
        """Build NetworkX graph with NVD-weighted edges"""
        self.G = nx.DiGraph()
        for node in self.nodes_set:
            self.G.add_node(node, ntype=self.node_types.get(node, "other"))
        for (u, v), count in self.edge_counts.items():
            avg_weight = self.edge_weights[(u, v)] / count if count else self.edge_weights[(u, v)]
            latest_metadata = self.edge_metadata.get((u, v), [{}])[-1]
            self.G.add_edge(
                u, v,
                weight=avg_weight,
                frequency=count,
                nvd_score=latest_metadata.get('nvd_score', 0.0),
                severity=latest_metadata.get('severity', 'none'),
                attack_types=latest_metadata.get('attack_types', [])
            )
        self.metadata['total_nodes'] = self.G.number_of_nodes()
        self.metadata['total_edges'] = self.G.number_of_edges()

    def compute_universal_shortest_paths(self):
        """Compute shortest paths with NVD-weighted edges"""
        if self.G.number_of_nodes() == 0:
            return {}, None
        source_scores = defaultdict(float)
        for node in self.G.nodes():
            for edge in self.G.edges(node, data=True):
                source_scores[node] += edge[2].get('nvd_score', 0.0)
        source_node = max(source_scores, key=source_scores.get) if source_scores else list(self.G.nodes())[0]
        try:
            paths = nx.single_source_dijkstra_path_length(self.G, source_node, weight='weight')
            return paths, source_node
        except Exception:
            return {}, source_node

    def universal_anomaly_detection(self, paths):
        """Universal anomaly detection using adaptive thresholding"""
        if not paths or len(paths) < 3:
            self.anomalies = []
            self.metadata['anomalies_detected'] = 0
            return
        distances = list(paths.values())
        self.adaptive_detector.update_data(distances)
        adaptive_threshold = self.adaptive_detector.compute_adaptive_threshold()
        self.metadata['adaptive_threshold'] = adaptive_threshold
        anomaly_indices, z_scores = self.adaptive_detector.detect_anomalies(distances)
        self.anomalies = []
        nodes = list(paths.keys())
        for idx in anomaly_indices:
            if idx < len(nodes):
                node = nodes[idx]
                z_score = z_scores[idx]
                attack_info = self.attack_intelligence.get(node, [])
                latest_attack = attack_info[-1] if attack_info else {}
                anomaly_info = {
                    'node': node,
                    'distance': distances[idx],
                    'z_score': z_score,
                    'adaptive_threshold': adaptive_threshold,
                    'node_type': self.node_types.get(node, 'unknown'),
                    'attack_types': latest_attack.get('attack_types', ['unknown']),
                    'nvd_score': latest_attack.get('score', 0.0),
                    'severity': latest_attack.get('severity', 'unknown'),
                    'description': self._generate_universal_description(node, distances[idx], z_score, latest_attack),
                    'confidence': min(1.0, abs(z_score) / adaptive_threshold) if adaptive_threshold else 0.0,
                    'timestamp': datetime.now().isoformat()
                }
                self.anomalies.append(anomaly_info)
        self.metadata['anomalies_detected'] = len(self.anomalies)
        print(f"Detected {len(self.anomalies)} anomalies using adaptive threshold: {adaptive_threshold:.2f}")

    def _generate_universal_description(self, node, distance, z_score, attack_info):
        """Generate universal anomaly description"""
        attack_types = attack_info.get('attack_types', ['unknown'])
        nvd_score = attack_info.get('score', 0.0)
        severity = attack_info.get('severity', 'unknown')
        base_desc = (f"Node '{node}' exhibits anomalous behavior "
                    f"(distance: {distance:.2f}, Z-score: {z_score:.2f}).")
        if attack_types != ['unknown']:
            attack_desc = f" Detected attack patterns: {', '.join(attack_types)}."
            if nvd_score > 0:
                attack_desc += f" NVD risk score: {nvd_score:.1f} ({severity} severity)."
        else:
            attack_desc = " No specific attack pattern identified."
        confidence_desc = f" Detection confidence: {min(100, abs(z_score) * 25):.0f}%."
        return base_desc + attack_desc + confidence_desc

    def create_universal_visualization(self):
        """Create universal graph visualization (plotly)"""
        if go is None:
            print("Plotly not available: can't create interactive visualization.")
            return None
        if self.G.number_of_nodes() == 0:
            fig = go.Figure()
            fig.add_annotation(text="No graph data available")
            return fig
        pos = nx.spring_layout(self.G, k=3, iterations=100, seed=42)
        node_data = self._prepare_node_data(pos)
        edge_data = self._prepare_edge_data(pos)
        fig = go.Figure()
        for edge_trace in edge_data:
            fig.add_trace(edge_trace)
        fig.add_trace(go.Scatter(
            x=node_data['x'], y=node_data['y'],
            mode='markers+text',
            text=node_data['labels'],
            textposition="middle center",
            hoverinfo='text',
            hovertext=node_data['hover_text'],
            marker=dict(
                size=node_data['sizes'],
                color=node_data['colors'],
                line=dict(width=2, color='black'),
                colorscale='RdYlBu_r',
                showscale=True,
                colorbar=dict(title="Risk Level")
            ),
            name='Nodes'
        ))
        fig.update_layout(
            title=f'Universal Log Analysis - {len(self.anomalies)} Anomalies Detected',
            showlegend=False,
            hovermode='closest',
            margin=dict(b=20, l=5, r=5, t=40),
            annotations=[dict(text="Red nodes indicate anomalies detected by NVD scoring",
                     showarrow=False, xref="paper", yref="paper",
                     x=0.005, y=-0.002, xanchor='left', yanchor='bottom',
                     font=dict(color="#888", size=12))],
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)
        )
        return fig

    def _prepare_node_data(self, pos):
        node_x, node_y, node_colors, node_sizes = [], [], [], []
        node_labels, hover_texts = [], []
        anomaly_nodes = {anomaly['node'] for anomaly in self.anomalies}
        for node in self.G.nodes():
            x, y = pos[node]
            node_x.append(x)
            node_y.append(y)
            risk_score = 0.0
            if node in self.attack_intelligence:
                attacks = self.attack_intelligence[node]
                risk_score = sum(attack.get('score', 0) for attack in attacks[-5:])
            degree = self.G.degree(node)
            size = 10 + (degree * 2) + (risk_score * 2)
            node_sizes.append(min(50, max(10, size)))
            if node in anomaly_nodes:
                node_colors.append(10.0)
                node_labels.append(f"⚠{node}")
            else:
                node_colors.append(risk_score)
                node_labels.append(node)
            ntype = self.node_types.get(node, 'unknown')
            hover_info = [f"Node: {node}", f"Type: {ntype}", f"Degree: {degree}"]
            if node in anomaly_nodes:
                anomaly = next(a for a in self.anomalies if a['node'] == node)
                hover_info.extend([
                    f"⚠ ANOMALY DETECTED",
                    f"Z-score: {anomaly['z_score']:.2f}",
                    f"NVD Score: {anomaly['nvd_score']:.1f}",
                    f"Severity: {anomaly['severity']}"
                ])
            if node in self.attack_intelligence:
                recent_attacks = self.attack_intelligence[node][-3:]
                hover_info.append("Recent attack patterns:")
                for attack in recent_attacks:
                    attack_types = ', '.join(attack['attack_types'])
                    hover_info.append(f"  • {attack_types} ({attack['severity']})")
            hover_texts.append('<br>'.join(hover_info))
        return {
            'x': node_x, 'y': node_y, 'colors': node_colors,
            'sizes': node_sizes, 'labels': node_labels, 'hover_text': hover_texts
        }

    def _prepare_edge_data(self, pos):
        if go is None:
            return []
        edge_traces = []
        severity_groups = defaultdict(list)
        for u, v, data in self.G.edges(data=True):
            severity = data.get('severity', 'none')
            nvd_score = data.get('nvd_score', 0.0)
            severity_groups[severity].append((u, v, nvd_score))
        severity_styles = {
            'critical': {'color': 'red', 'width': 4},
            'high': {'color': 'orange', 'width': 3},
            'medium': {'color': 'yellow', 'width': 2},
            'low': {'color': 'lightblue', 'width': 1},
            'none': {'color': 'lightgray', 'width': 0.5}
        }
        for severity, edges in severity_groups.items():
            if not edges:
                continue
            edge_x, edge_y = [], []
            for u, v, nvd_score in edges:
                x0, y0 = pos[u]
                x1, y1 = pos[v]
                edge_x.extend([x0, x1, None])
                edge_y.extend([y0, y1, None])
            style = severity_styles.get(severity, severity_styles['none'])
            trace = go.Scatter(
                x=edge_x, y=edge_y,
                mode='lines',
                line=dict(width=style['width'], color=style['color']),
                hoverinfo='none',
                name=f'{severity.title()} Risk'
            )
            edge_traces.append(trace)
        return edge_traces

    def create_intelligence_dashboard(self):
        """Create comprehensive intelligence dashboard"""
        graph_fig = self.create_universal_visualization() if go is not None else None
        attack_timeline = self._create_attack_timeline() if go is not None else None
        risk_matrix = self._create_risk_matrix() if go is not None else None
        threshold_history = self._create_threshold_history() if make_subplots is not None else None
        return graph_fig, attack_timeline, risk_matrix, threshold_history

    def _create_attack_timeline(self):
        if not any(self.attack_intelligence.values()):
            fig = go.Figure()
            fig.add_annotation(text="No attack intelligence available")
            return fig
        all_attacks = []
        for node, attacks in self.attack_intelligence.items():
            for attack in attacks:
                attack_copy = attack.copy()
                attack_copy['node'] = node
                all_attacks.append(attack_copy)
        if not all_attacks:
            fig = go.Figure()
            fig.add_annotation(text="No attack data to display")
            return fig
        df = pd.DataFrame(all_attacks)
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df['hour'] = df['timestamp'].dt.floor('H')
        hourly_attacks = df.groupby(['hour', 'severity']).size().reset_index(name='count')
        fig = go.Figure()
        colors = {'critical': 'red', 'high': 'orange', 'medium': 'yellow', 'low': 'lightblue', 'none': 'gray'}
        for severity in hourly_attacks['severity'].unique():
            severity_data = hourly_attacks[hourly_attacks['severity'] == severity]
            fig.add_trace(go.Scatter(
                x=severity_data['hour'],
                y=severity_data['count'],
                mode='lines+markers',
                name=f'{severity.title()} Severity',
                line=dict(color=colors.get(severity, 'gray'))
            ))
        fig.update_layout(
            title='Attack Intelligence Timeline',
            xaxis_title='Time',
            yaxis_title='Attack Count',
            hovermode='x unified'
        )
        return fig

    def _create_risk_matrix(self):
        if not self.anomalies:
            fig = go.Figure()
            fig.add_annotation(text="No risk data available")
            return fig
        nodes = [a['node'] for a in self.anomalies]
        nvd_scores = [a['nvd_score'] for a in self.anomalies]
        z_scores = [abs(a['z_score']) for a in self.anomalies]
        fig = go.Figure(data=go.Scatter(
            x=nvd_scores,
            y=z_scores,
            mode='markers+text',
            text=nodes,
            textposition="top center",
            marker=dict(
                size=[20 + a['confidence'] * 30 for a in self.anomalies],
                color=nvd_scores,
                colorscale='Reds',
                showscale=True,
                colorbar=dict(title="NVD Risk Score")
            ),
            hovertemplate='<b>%{text}</b><br>NVD Score: %{x}<br>Z-Score: %{y}<extra></extra>'
        ))
        fig.update_layout(
            title='Risk Assessment Matrix',
            xaxis_title='NVD Risk Score',
            yaxis_title='Statistical Anomaly Score (|Z-Score|)',
        )
        fig.add_shape(type="rect", x0=0, y0=0, x1=5, y1=2, fillcolor="green", opacity=0.1, layer="below", line_width=0)
        fig.add_shape(type="rect", x0=5, y0=2, x1=8, y1=4, fillcolor="yellow", opacity=0.1, layer="below", line_width=0)
        fig.add_shape(type="rect", x0=8, y0=4, x1=10, y1=10, fillcolor="red", opacity=0.1, layer="below", line_width=0)
        return fig

    def _create_threshold_history(self):
        if not self.adaptive_detector.threshold_history:
            fig = go.Figure()
            fig.add_annotation(text="No threshold history available")
            return fig
        df = pd.DataFrame(self.adaptive_detector.threshold_history)
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                           subplot_titles=['Adaptive Threshold Evolution', 'Sample Size'])
        fig.add_trace(go.Scatter(
            x=df['timestamp'],
            y=df['threshold'],
            mode='lines+markers',
            name='Adaptive Threshold'
        ), row=1, col=1)
        fig.add_trace(go.Scatter(
            x=df['timestamp'],
            y=df['sample_size'],
            mode='lines',
            name='Sample Size'
        ), row=2, col=1)
        fig.update_layout(title='Threshold Adaptation Over Time')
        fig.update_xaxes(title_text="Time", row=2, col=1)
        fig.update_yaxes(title_text="Threshold", row=1, col=1)
        fig.update_yaxes(title_text="Samples", row=2, col=1)
        return fig

    def save_comprehensive_metadata(self, filename="universal_log_analysis.json"):
        """Save comprehensive analysis metadata"""
        self.metadata.update({
            'last_updated': datetime.now().isoformat(),
            'nvd_queries': len(self.nvd_scorer.vulnerability_cache) if self.nvd_scorer else 0,
            'auto_discovered_patterns': len(self.auto_patterns),
            'total_processed_logs': len(self.processed_logs)
        })
        comprehensive_data = {
            'metadata': self.metadata,
            'anomalies': self.anomalies,
            'attack_intelligence': dict(self.attack_intelligence),
            'threshold_history': self.adaptive_detector.threshold_history,
            'node_types_distribution': dict(Counter(self.node_types.values())),
            'nvd_cache_stats': {
                'total_cached': len(self.nvd_scorer.vulnerability_cache) if self.nvd_scorer else 0,
                'cache_file': self.nvd_scorer.cache_file if self.nvd_scorer else None
            },
            'auto_discovered_patterns': list(self.auto_patterns),
            'processed_logs_sample': self.processed_logs[-10:] if self.processed_logs else []
        }
        try:
            with open(filename, 'w') as f:
                json.dump(comprehensive_data, f, indent=2, default=str)
            if self.nvd_scorer:
                self.nvd_scorer.save_cache()
            print(f"Comprehensive metadata saved to {filename}")
        except Exception as e:
            print(f"Failed to save metadata: {e}")

    def run_universal_analysis(self, log_data, log_type='auto'):
        """Run universal analysis on any log format"""
        start_time = time.time()
        print("Starting universal log analysis...")
        if isinstance(log_data, str):
            if os.path.exists(log_data):
                with open(log_data, 'r', encoding='utf-8', errors='ignore') as f:
                    log_lines = [line.strip() for line in f if line.strip()]
            else:
                log_lines = [line.strip() for line in log_data.split('\n') if line.strip()]
        elif isinstance(log_data, list):
            log_lines = [str(line).strip() for line in log_data if str(line).strip()]
        else:
            raise ValueError("log_data must be file path, log string, or list of log lines")
        print(f"Processing {len(log_lines)} log lines...")
        if log_type == 'auto':
            self.auto_discover_patterns(log_lines)
            print(f"Auto-discovered {len(self.auto_patterns)} new patterns")
        for i, line in enumerate(log_lines):
            try:
                self.process_log_line(line)
            except Exception:
                continue
            if i % 1000 == 0 and i > 0:
                print(f"Processed {i} lines...")
        print("Building graph with NVD-weighted edges...")
        self.build_graph()
        print("Computing shortest paths...")
        paths, source_node = self.compute_universal_shortest_paths()
        print("Running universal anomaly detection...")
        self.universal_anomaly_detection(paths)
        self.metadata['processing_time'] = time.time() - start_time
        self.save_comprehensive_metadata()
        print(f"Analysis complete in {self.metadata['processing_time']:.2f} seconds!")
        print(f"Detected {self.metadata['anomalies_detected']} anomalies")
        if self.metadata.get('adaptive_threshold') is not None:
            print(f"Adaptive threshold: {self.metadata['adaptive_threshold']:.2f}")
        return self.create_intelligence_dashboard()


# ---------------------------
# Jupyter/Colab helper: interface creator
# ---------------------------
def create_universal_interface():
    """Create universal Jupyter interface"""
    try:
        import ipywidgets as widgets
        from IPython.display import display, clear_output, HTML
    except Exception:
        print("ipywidgets not available; interactive GUI cannot be created.")
        return None, None

    analyzer = UniversalLogAnalyzer()
    log_input = widgets.Textarea(
        value="",
        placeholder="Paste log data here or specify file path",
        description="Log Data:",
        style={'description_width': 'initial'},
        layout={'width': '100%', 'height': '200px'}
    )
    file_upload = widgets.FileUpload(
        accept='.log,.txt,.csv',
        multiple=False,
        description="Or upload file:"
    )
    log_type = widgets.Dropdown(
        options=[('Auto-detect', 'auto'), ('Syslog', 'syslog'), ('Custom', 'custom')],
        value='auto',
        description='Log Type:',
        style={'description_width': 'initial'}
    )
    enable_nvd = widgets.Checkbox(
        value=True,
        description="Enable NVD scoring (requires internet)",
        style={'description_width': 'initial'}
    )
    realtime_mode = widgets.Checkbox(
        value=False,
        description="Enable real-time monitoring",
        style={'description_width': 'initial'}
    )
    analyze_button = widgets.Button(
        description="🔍 Run Universal Analysis",
        button_style='success',
        layout={'width': '200px'}
    )
    output_area = widgets.Output()
    status_label = widgets.HTML("<b>Status:</b> Ready")

    def run_universal_analysis(button):
        with output_area:
            clear_output(wait=True)
            status_label.value = "<b>Status:</b> Running analysis..."
            try:
                log_data = log_input.value.strip()
                if file_upload.value:
                    uploaded_file = list(file_upload.value.values())[0]
                    log_data = uploaded_file['content'].decode('utf-8')
                    print(f"Using uploaded file: {uploaded_file['metadata']['name']}")
                if not log_data:
                    print("Please provide log data or upload a file")
                    status_label.value = "<b>Status:</b> No data provided"
                    return
                if not enable_nvd.value:
                    print("NVD scoring disabled - using fallback scoring")
                    analyzer.nvd_scorer = None
                dashboard = analyzer.run_universal_analysis(log_data, log_type.value)
                graph_fig, timeline_fig, risk_fig, threshold_fig = dashboard
                print("\n" + "="*60)
                print("UNIVERSAL LOG ANALYSIS RESULTS")
                print("="*60)
                print(f"Total nodes: {analyzer.metadata['total_nodes']}")
                print(f"Total edges: {analyzer.metadata['total_edges']}")
                print(f"Anomalies detected: {analyzer.metadata['anomalies_detected']}")
                print(f"Adaptive threshold: {analyzer.metadata.get('adaptive_threshold', 'N/A')}")
                print(f"Processing time: {analyzer.metadata['processing_time']:.2f}s")
                if graph_fig:
                    graph_fig.show()
                if timeline_fig:
                    timeline_fig.show()
                if risk_fig:
                    risk_fig.show()
                if threshold_fig:
                    threshold_fig.show()
                if analyzer.anomalies:
                    print("\n🚨 DETECTED ANOMALIES:")
                    for i, anomaly in enumerate(analyzer.anomalies, 1):
                        print(f"\n{i}. Node: {anomaly['node']}")
                        print(f"   Type: {anomaly['node_type']}")
                        print(f"   Severity: {anomaly['severity']}")
                        print(f"   NVD Score: {anomaly['nvd_score']:.1f}")
                        print(f"   Z-Score: {anomaly['z_score']:.2f}")
                        print(f"   Attack Types: {', '.join(anomaly['attack_types'])}")
                        print(f"   Confidence: {anomaly['confidence']*100:.0f}%")
                        print(f"   Description: {anomaly['description']}")
                status_label.value = f"<b>Status:</b> ✅ Analysis complete ({len(analyzer.anomalies)} anomalies found)"
            except Exception as e:
                print(f"Error during analysis: {str(e)}")
                status_label.value = f"<b>Status:</b> ❌ Error: {str(e)[:50]}..."

    analyze_button.on_click(run_universal_analysis)
    interface = widgets.VBox([
        widgets.HTML("""
        <h2>🛡️ Universal Log Anomaly Detection System</h2>
        <p><b>Features:</b> NVD Integration, Adaptive Thresholding, Multi-format Support</p>
        <hr>
        """),
        widgets.HBox([
            widgets.VBox([log_input, file_upload], layout={'width': '60%'}),
            widgets.VBox([log_type, enable_nvd, realtime_mode], layout={'width': '40%'})
        ]),
        widgets.HBox([analyze_button, status_label]),
        output_area
    ])
    return interface, analyzer

# ---------------------------
# DEMO / AUTO-RUN wrapper
# ---------------------------
def _auto_run_if_log_present():
    analyzer = UniversalLogAnalyzer()
    # prefer /mnt/data/syslogs.log if present (as user uploaded)
    candidate_paths = [
        "/mnt/data/syslogs.log",
        "syslogs.log",
        "/content/syslogs.log"
    ]
    existing = None
    for p in candidate_paths:
        if os.path.exists(p):
            existing = p
            break
    if existing:
        print(f"Found log file: {existing}. Running analysis...")
        dashboard = analyzer.run_universal_analysis(existing)
        graph_fig, timeline_fig, risk_fig, threshold_fig = dashboard
        try:
            if graph_fig:
                graph_fig.show()
            if timeline_fig:
                timeline_fig.show()
            if risk_fig:
                risk_fig.show()
            if threshold_fig:
                threshold_fig.show()
        except Exception:
            pass
        try:
            # Try to save JSON report to working folder
            analyzer.save_comprehensive_metadata("analysis_results.json")
            print("Saved analysis_results.json in the working directory.")
        except Exception:
            pass
        return analyzer
    else:
        # If running in Colab, offer upload; else instruct user
        try:
            from google.colab import files
            print("No pre-existing syslogs file found. Please upload one (syslogs.log).")
            uploaded = files.upload()
            if uploaded:
                key = list(uploaded.keys())[0]
                print(f"Uploaded {key}. Running analysis...")
                analyzer = UniversalLogAnalyzer()
                dashboard = analyzer.run_universal_analysis(key)
                graph_fig, timeline_fig, risk_fig, threshold_fig = dashboard
                try:
                    if graph_fig:
                        graph_fig.show()
                    if timeline_fig:
                        timeline_fig.show()
                    if risk_fig:
                        risk_fig.show()
                    if threshold_fig:
                        threshold_fig.show()
                except Exception:
                    pass
                try:
                    analyzer.save_comprehensive_metadata("analysis_results.json")
                    files.download("analysis_results.json")
                except Exception:
                    pass
                return analyzer
        except Exception:
            # Not in Colab; print instructions
            print("No syslogs file found at /mnt/data/syslogs.log or working dir.")
            print("To run analysis, either:")
            print("1) Upload your logs to the notebook and set log_path to that file path;")
            print("2) Or call create_universal_interface() and paste logs into the GUI.")
            return None

# Run auto-run helper when this script executes
_analyzer_instance = _auto_run_if_log_present()

# Also expose analyzer variable for interactive use
analyzer = _analyzer_instance

# Instructions for interactive usage if auto-run didn't process anything
print("\nIf nothing ran automatically above, you can:")
print(" - Use create_universal_interface() to open an interactive GUI (requires ipywidgets).")
print(" - Or run: analyzer = UniversalLogAnalyzer(); analyzer.run_universal_analysis('/path/to/your/syslogs.log')")
print(" - After run, download results with analyzer.save_comprehensive_metadata('analysis_results.json')")

# ---------------------------
# END OF FILE
# ---------------------------

Found log file: syslogs.log. Running analysis...
Starting universal log analysis...
Processing 109 log lines...
Auto-discovered 9 new patterns
Building graph with NVD-weighted edges...
Computing shortest paths...
Running universal anomaly detection...
Detected 2 anomalies using adaptive threshold: 2.00
Comprehensive metadata saved to universal_log_analysis.json
Analysis complete in 5.43 seconds!
Detected 2 anomalies
Adaptive threshold: 2.00


Comprehensive metadata saved to analysis_results.json
Saved analysis_results.json in the working directory.

If nothing ran automatically above, you can:
 - Use create_universal_interface() to open an interactive GUI (requires ipywidgets).
 - Or run: analyzer = UniversalLogAnalyzer(); analyzer.run_universal_analysis('/path/to/your/syslogs.log')
 - After run, download results with analyzer.save_comprehensive_metadata('analysis_results.json')
